# Lab 07 · Memory, caching and checkpoints

**Part A** (about 45 min): compute the memory regions, find them in the Spark UI, cause a spill, and compare cache, checkpoint and a driver limit.<br>
**Part B** (about 70 min): 11 problems with medians, percentiles, rollups, `max_by`, shares and snapshots.

**Checklist questions Part A gives you evidence for:**<br>
<span style="color:#d93025"><b>5.1</b> The executor's memory regions · <b>5.2</b> Spilling to disk · <b>5.3</b> spark.memory.fraction · <b>5.4</b> Off-heap memory · <b>5.5</b> Fixing a driver OOM · <b>5.6</b> User memory · <b>5.7</b> GC overhead · <b>5.8</b> When to unpersist() · <b>5.9</b> The risk of high spark.executor.cores · <b>5.10</b> Unified memory management · <b>5.11</b> checkpoint, localCheckpoint, cache and persist · <b>6.10</b> Why limit checkpoint()</span>

## How to work
* Start a **fresh kernel**, then run the **Setup** cell: the driver memory (1 GB) is only applied when Spark starts. `F` is `pyspark.sql.functions`, `Window` is `pyspark.sql.Window`.
* **Part A**: run the code under the question (or add a line), look at the output or the Spark UI, and write your answer as a comment.
* **Part B**: write the code yourself. Your result must look exactly like the **expected output**, with the rows in the same order.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.

## Documentation
* [Memory management overview](https://spark.apache.org/docs/latest/tuning.html#memory-management-overview): execution and storage, unified memory
* [Memory configuration](https://spark.apache.org/docs/latest/configuration.html#memory-management): `spark.memory.fraction`, `spark.memory.storageFraction`, off-heap
* [Garbage collection tuning](https://spark.apache.org/docs/latest/tuning.html#garbage-collection-tuning)
* [RDD persistence](https://spark.apache.org/docs/latest/rdd-programming-guide.html#rdd-persistence): storage levels, removing data
* [DataFrame.checkpoint](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.checkpoint.html) and [localCheckpoint](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.localCheckpoint.html)
* [spark.driver.maxResultSize](https://spark.apache.org/docs/latest/configuration.html#application-properties)
* [GroupedData](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/grouping.html) and [DataFrame.rollup](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.rollup.html)
* [Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html): `median`, `percentile`, `max_by`, `collect_list`, `array_join`, `eqNullSafe`

## The data
Three small tables of an online shop. The setup cell creates them, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date* (the day the customer registered)
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know:
* order 1011 is a guest checkout: it has no *customer_id*
* customer 6 has no email and no city
* customer 7 has never ordered, and product 107 has never been sold
* order 1016 is for product 108, which is no longer in *products*

For this lab the setup cell also makes:
* **codes** (Part A): 3,000,000 rows: *id* and a 64-character text *code*
* **customers_new** (Part B): the customers table one week later: some rows changed, one customer is gone and one is new

## Setup
Run this cell once, in a fresh kernel. It starts Spark with a 1 GB driver, saves the three shop tables as Parquet files in `data/shop/`, and defines *codes*, *customers_new* and the helpers `noop`, `spill_report` and `plan_lines`.

In [ ]:
# Setup: starts Spark and creates the shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder
         .master("local[4]")   # 4 tasks at a time on your machine
         .appName("lab07")
         .config("spark.sql.adaptive.enabled", "false")        # AQE off: question A3 sets the partitions itself
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.driver.memory", "1g")                 # question A2 reads this number
         .config("spark.driver.maxResultSize", "10m")         # question A7: a small result limit
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")                  # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

def noop(df):
    """Runs the whole query but keeps no result (Spark's "noop" sink)."""
    df.write.format("noop").mode("overwrite").save()

import json, urllib.request

def spark_api(path):
    """Reads the Spark UI's REST API, e.g. spark_api("/stages")."""
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

def spill_report(action):
    """Runs action() and returns its time, how much its stages spilled, and their garbage collection time."""
    before = {s["stageId"] for s in spark_api("/stages")}
    start = time.time()
    action()
    seconds = time.time() - start
    time.sleep(0.5)
    new_stages = [s for s in spark_api("/stages") if s["stageId"] not in before]
    mb = lambda key: round(sum(s[key] for s in new_stages) / 1e6)
    return (f"{seconds:.1f} s | spilled {mb('memoryBytesSpilled')} MB (memory), {mb('diskBytesSpilled')} MB (disk) "
            f"| GC {sum(s['jvmGcTime'] for s in new_stages)} ms")

def plan_lines(df):
    """How many lines the logical plan of df has: a measure of how long its lineage is."""
    return len(df._jdf.queryExecution().analyzed().toString().splitlines())

# Part A: 3,000,000 rows with a long text code (64 characters)
codes = spark.range(3_000_000).select("id", F.sha2(F.col("id").cast("string"), 256).alias("code"))

# Part B: the customers table one week later
customers_new = spark.createDataFrame([
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Munich",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     "farid@mail.az",    None,       date(2024, 3, 10)),
    (8, "Hana Sato",        "hana@mail.jp",     "Osaka",    date(2024, 4, 2)),
], customers.schema)

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the tables:

In [ ]:
customers.show()
customers_new.show()
products.show()
orders.show()
codes.show(3, truncate=False)

## Part A · Memory, spills and persistence

### The executor's memory <span style="color:#d93025">(5.1, 5.3, 5.4, 5.6, 5.10)</span>

**A1. Finish `regions(heap_mb)` with the rules in the cell: it returns the usable, unified, protected storage and user memory of an executor heap. What are they for a 4096 MB heap?**

In [ ]:
# My answer:
# 300 MB are reserved for Spark; usable = heap - 300
# unified = usable * spark.memory.fraction (0.6); protected storage = unified * spark.memory.storageFraction (0.5)
# user = usable * (1 - 0.6)
def regions(heap_mb):
    """(usable, unified, protected storage, user) memory of an executor heap, in MB."""
    usable = heap_mb - 300
    return ...

regions(4096)

**A2. This session's driver has a 1 GB heap (in local mode the driver is also the executor). Open the Spark UI → Executors tab: what is the total of *Storage Memory*? Compare it with `regions(1024)`.**

In [ ]:
# My answer:
regions(1024)

### Spills, GC and cores <span style="color:#d93025">(5.2, 5.7, 5.9)</span>

**A3. Sort the 3 million codes with 1, 4 and 64 shuffle partitions. How much spills to disk each time?**

In [ ]:
# My answer:
for partitions in [1, 4, 64]:
    spark.conf.set("spark.sql.shuffle.partitions", partitions)
    print(f"{partitions:2} partitions:", spill_report(lambda: noop(codes.orderBy("code"))))

**A4. On an executor with an 8192 MB heap, how much execution memory can one task count on with 4 cores, and with 16 cores? Use `regions`.**

In [ ]:
# My answer:
# unified = regions(8192)[1]      (your function from A1)

### cache, persist and checkpoint <span style="color:#d93025">(5.8, 5.11, 6.10)</span>

**A5. `report_data` is used by three reports. Time them without cache, then after `report_data.cache()` (run them twice). Find the cached data in the Spark UI's Storage tab, then remove it with `unpersist()`.**

In [ ]:
# My answer:
report_data = (codes.withColumn("bucket", F.col("id") % 100)
                    .groupBy("bucket").agg(F.count("*").alias("n"), F.max("code").alias("max_code")))

def three_reports(df):
    df.agg(F.sum("n")).collect()
    df.filter("n > 0").count()
    df.orderBy("bucket").first()

start = time.time(); three_reports(report_data); print(f"no cache:          {time.time() - start:.2f} s")

**A6. Build a DataFrame with 60 `withColumn` steps. How many lines does its plan have before and after `checkpoint()`? What does the checkpoint write to disk?**

In [ ]:
# My answer:
long_df = spark.range(1000)
for i in range(60):
    long_df = long_df.withColumn(f"c{i}", F.col("id") + i)
spark.sparkContext.setCheckpointDir("data/shop/checkpoints")
print(plan_lines(long_df), "plan lines before")

### The driver <span style="color:#d93025">(5.5)</span>

**A7. This session allows the driver at most 10 MB of results (`spark.driver.maxResultSize`). Collect 3 million rows: what happens? Then get an answer about the rows without collecting them.**

In [ ]:
# My answer:
rows = spark.range(3_000_000).selectExpr("id", "repeat('x', 20) AS padding")
rows.collect()

**Done with Part A?** Explain each red checklist question at the top aloud, in your own words, using what you saw.

## Part B · Aggregation problems
11 problems on the shop tables and *customers_new*. Where an expected output is left-aligned, print with `.show(truncate=False)`.

**B1. What is the median quantity of each status? Show status and median_quantity.**

Expected output:

```
+---------+---------------+
|   status|median_quantity|
+---------+---------------+
|cancelled|            1.5|
|delivered|            1.0|
|  shipped|            1.5|
+---------+---------------+
```

<details><summary>Hint</summary>

`F.median(column)`; with an even number of values it is the average of the two middle ones.

</details>

**B2. Show the 25th, 50th and 75th percentile of the product prices, as one row with the columns p25, p50 and p75.**

Expected output:

```
+----+----+-----+
| p25| p50|  p75|
+----+----+-----+
|32.5|60.0|117.5|
+----+----+-----+
```

<details><summary>Hint</summary>

`F.percentile(column, 0.25)` is exact; `F.percentile_approx` is faster on big data.

</details>

**B3. Show the revenue of delivered orders by category and product, with a subtotal for each category and a grand total (empty names mark the subtotals). Sort by category and product, with subtotals after their products.**

Expected output:

```
+-----------+------------+-------+
|   category|product_name|revenue|
+-----------+------------+-------+
|      Books|    Cookbook|   25.0|
|      Books|       Novel|  180.0|
|      Books|        NULL|  205.0|
|Electronics|  Headphones|  450.0|
|Electronics|      Laptop| 2400.0|
|Electronics|        NULL| 2850.0|
|    Kitchen|     Blender|   60.0|
|    Kitchen|        NULL|   60.0|
|       NULL|        NULL| 3115.0|
+-----------+------------+-------+
```

<details><summary>Hint</summary>

`rollup("category", "product_name")` adds the subtotal rows; `asc_nulls_last()` puts them after the details.

</details>

**B4. For each customer, list their order ids as one text, smallest first, like `1001, 1002, 1007, 1015`. Leave out the guest order. Show customer_id and order_ids.**

Expected output:

```
+-----------+----------------------+
|customer_id|order_ids             |
+-----------+----------------------+
|1          |1001, 1002, 1007, 1015|
|2          |1003, 1005, 1013      |
|3          |1004, 1009, 1016      |
|4          |1006, 1010            |
|5          |1008, 1014            |
|6          |1012                  |
+-----------+----------------------+
```

<details><summary>Hint</summary>

`F.collect_list` gathers the values into an array, `F.array_sort` sorts it, `F.array_join(array, ", ")` glues it into text.

</details>

**B5. Show the product of each customer's latest order: customer_id and latest_product_id (leave out the guest order).**

Expected output:

```
+-----------+-----------------+
|customer_id|latest_product_id|
+-----------+-----------------+
|          1|              103|
|          2|              101|
|          3|              108|
|          4|              106|
|          5|              105|
|          6|              103|
+-----------+-----------------+
```

<details><summary>Hint</summary>

`F.max_by("product_id", "order_date")` returns the product_id of the row with the largest order_date.

</details>

**B6. What share of the delivered revenue does each category bring, in percent (1 decimal)? Show category, revenue and share_pct, highest first.**

Expected output:

```
+-----------+-------+---------+
|   category|revenue|share_pct|
+-----------+-------+---------+
|Electronics| 2850.0|     91.5|
|      Books|  205.0|      6.6|
|    Kitchen|   60.0|      1.9|
+-----------+-------+---------+
```

<details><summary>Hint</summary>

The total of all rows is `F.sum("revenue").over(Window.partitionBy())`, a window over everything.

</details>

**B7. Show the second most expensive product of each category (categories with only one product drop out). Columns: category, product_name and price.**

Expected output:

```
+-----------+------------+-----+
|   category|product_name|price|
+-----------+------------+-----+
|      Books|       Novel| 15.0|
|Electronics|  Headphones|150.0|
|    Kitchen|     Blender| 60.0|
+-----------+------------+-----+
```

<details><summary>Hint</summary>

The classic Nth highest: `dense_rank` per category, keep rank 2.

</details>

**B8. Which products cost more than the average price of their category? Show category, product_name, price and category_avg (2 decimals).**

Expected output:

```
+-----------+------------+------+------------+
|   category|product_name| price|category_avg|
+-----------+------------+------+------------+
|      Books|    Cookbook|  25.0|        20.0|
|Electronics|      Laptop|1200.0|       675.0|
|    Kitchen|Coffee Maker|  85.0|        72.5|
+-----------+------------+------+------------+
```

<details><summary>Hint</summary>

The category average as a window column, then a filter.

</details>

**B9. For each city, count the customers and how many of them have an email. Customers without a city count under `Unknown`. Show city, customers and with_email.**

Expected output:

```
+--------+---------+----------+
|    city|customers|with_email|
+--------+---------+----------+
|  Berlin|        2|         2|
|  London|        1|         1|
|  Madrid|        1|         1|
|Tashkent|        2|         2|
| Unknown|        1|         0|
+--------+---------+----------+
```

<details><summary>Hint</summary>

`F.count("email")` counts only the rows where email is not null.

</details>

**B10. Compare `customers` with `customers_new`: which customers were `added`, `removed` or `changed` (a different name, email or city)? Show customer_id and change.**

Expected output:

```
+-----------+-------+
|customer_id| change|
+-----------+-------+
|          2|changed|
|          6|changed|
|          7|removed|
|          8|  added|
+-----------+-------+
```

<details><summary>Hint</summary>

A full outer join on customer_id. Compare with `eqNullSafe`: with `==`, null compared to anything is null, so a missing email that got a value would not count as changed.

</details>

**B11. Make a table of delivered revenue with one row per category and one column per month (Jan, Feb, Mar), with 0 where nothing was sold.**

Expected output:

```
+-----------+------+-----+------+
|   category|   Jan|  Feb|   Mar|
+-----------+------+-----+------+
|      Books|  45.0| 25.0| 135.0|
|Electronics|1500.0|150.0|1200.0|
|    Kitchen|   0.0|  0.0|  60.0|
+-----------+------+-----+------+
```

<details><summary>Hint</summary>

`F.date_format(date, "MMM")` gives `Jan`; then `pivot("month", ["Jan", "Feb", "Mar"])` with a sum.

</details>

---
**Done?** In the Theory part of Day 7, answer the red checklist questions aloud.

## Answers
Try each question yourself first. Click a question to open its answer; Part A answers also show the output of the reference run.

<details>
<summary><b>A1</b> &nbsp;Finish regions(heap_mb) with the rules in the cell: it returns the usable, unified, protected storage and ...</summary>

**(3796, 2277.6, 1138.8, 1518.4).** The reserved 300 MB are for Spark itself. **Unified memory** (60% of the rest, `spark.memory.fraction`) is one pool shared by **execution** (shuffles, joins, sorts, aggregations) and **storage** (cache, broadcast variables). Either side can borrow the other's free space, and execution can even evict cached blocks, but not below the protected storage part (50% of unified, `spark.memory.storageFraction`): that is **unified memory management** (since Spark 1.6). **User memory**, the other 40%, holds your own objects, UDF data and Spark's internal metadata. **Off-heap memory** (`spark.memory.offHeap.enabled` and `spark.memory.offHeap.size`) is extra memory outside the Java heap: no garbage collection, used by Tungsten for execution and storage.

```python
def regions(heap_mb):
    """(usable, unified, protected storage, user) memory of an executor heap, in MB."""
    usable = heap_mb - 300
    unified = usable * 0.6
    return round(usable, 1), round(unified, 1), round(unified * 0.5, 1), round(usable * 0.4, 1)

regions(4096)
```

Result:

```
(3796, 2277.6, 1138.8, 1518.4)
```

</details>

<details>
<summary><b>A2</b> &nbsp;This session's driver has a 1 GB heap (in local mode the driver is also the executor). Open the Spark UI → ...</summary>

**434.4 MiB, exactly the unified memory of a 1024 MB heap** ((1024 − 300) × 0.6). The UI calls it storage memory because it shows how much could be used for caching, but execution uses the same pool.

```python
regions(1024)
```

Result:

```
(724, 434.4, 217.2, 289.6)
```

</details>

<details>
<summary><b>A3</b> &nbsp;Sort the 3 million codes with 1, 4 and 64 shuffle partitions. How much spills to disk each time?</summary>

**With 1 and 4 partitions about 200 MB spill to disk; with 64, nothing.** A task sorts its partition in execution memory; when the partition doesn't fit, Spark writes sorted pieces to disk and merges them later: **spilling**. The query still works, but disk is slow (compare the times). More partitions make smaller partitions that fit in memory. 4 partitions still spill because the 4 tasks run at the same time and share the unified memory, so each gets about a quarter. In the Spark UI, the Stages tab shows *Spill (Memory)* and *Spill (Disk)*, and *GC Time*: time the JVM spends freeing objects instead of running your tasks. High GC time (**GC overhead**) means too many objects for the heap: give it more memory, use fewer cores per executor, or cache in serialized form.

```python
for partitions in [1, 4, 64]:
    spark.conf.set("spark.sql.shuffle.partitions", partitions)
    print(f"{partitions:2} partitions:", spill_report(lambda: noop(codes.orderBy("code"))))
```

Result:

```
 1 partitions: 3.5 s | spilled 260 MB (memory), 206 MB (disk) | GC 70 ms
 4 partitions: 2.0 s | spilled 268 MB (memory), 206 MB (disk) | GC 120 ms
64 partitions: 1.9 s | spilled 0 MB (memory), 0 MB (disk) | GC 124 ms
```

*Your times will differ.*

</details>

<details>
<summary><b>A4</b> &nbsp;On an executor with an 8192 MB heap, how much execution memory can one task count on with 4 cores, and with ...</summary>

**About 1184 MB with 4 cores, but only about 296 MB with 16 cores.** All tasks running at the same time on an executor share its unified memory: with N running tasks, each one gets between 1/(2N) and 1/N of the execution pool. That is the risk of a **high `spark.executor.cores`**: more parallel tasks, less memory for each, so more spills and OOM errors, and more garbage collection. 4 to 5 cores per executor is a common choice.

```python
unified = regions(8192)[1]
print(round(unified / 4, 1), "MB per task with 4 cores")
print(round(unified / 16, 1), "MB per task with 16 cores")
```

Result:

```
1183.8 MB per task with 4 cores
295.9 MB per task with 16 cores
```

</details>

<details>
<summary><b>A5</b> &nbsp;report_data is used by three reports. Time them without cache, then after report_data.cache() (run them ...</summary>

**The first cached run is still slow (it fills the cache); the second is many times faster.** `cache()` is lazy: it only marks the DataFrame, the first action computes it and keeps its partitions in memory, and the next actions read them. The Storage tab lists the cached data and its size. Call **`unpersist()`** as soon as you no longer need it: cached blocks take memory that execution could use, and Spark only drops them by itself when it runs short of memory (least recently used first). `persist(level)` is the same with a choice of storage level: `MEMORY_AND_DISK` (the default for DataFrames), `MEMORY_ONLY`, `DISK_ONLY`, ...

```python
report_data = (codes.withColumn("bucket", F.col("id") % 100)
                    .groupBy("bucket").agg(F.count("*").alias("n"), F.max("code").alias("max_code")))

def three_reports(df):
    df.agg(F.sum("n")).collect()
    df.filter("n > 0").count()
    df.orderBy("bucket").first()

start = time.time(); three_reports(report_data); print(f"no cache:          {time.time() - start:.2f} s")
report_data.cache()
start = time.time(); three_reports(report_data); print(f"cache, first run:  {time.time() - start:.2f} s")
start = time.time(); three_reports(report_data); print(f"cache, second run: {time.time() - start:.2f} s")
print("cached:", report_data.is_cached)
report_data.unpersist()
print("cached after unpersist:", report_data.is_cached)
```

Result:

```
no cache:          1.46 s
cache, first run:  0.88 s
cache, second run: 0.24 s
cached: True
cached after unpersist: False
```

*Your times will differ.*

</details>

<details>
<summary><b>A6</b> &nbsp;Build a DataFrame with 60 withColumn steps. How many lines does its plan have before and after checkpoint()? ...</summary>

**61 lines before, 1 after (`Scan ExistingRDD`), and the checkpoint folder now holds data files.** `checkpoint()` computes the DataFrame, **writes it to reliable storage** (the checkpoint folder; HDFS or S3 on a cluster) and returns a DataFrame that reads those files: the lineage is cut, so the plan stays small, and a lost partition is read again instead of recomputed. `localCheckpoint()` keeps the data on the executors instead: faster, but lost if an executor dies. `cache()` and `persist()` keep the lineage (Spark can recompute lost cached blocks) and are the cheap way to reuse data. Use checkpoint sparingly: each one writes all the data and runs an extra job. It pays off for very long lineages (loops, iterative algorithms) or before steps that would be expensive to recompute.

```python
long_df = spark.range(1000)
for i in range(60):
    long_df = long_df.withColumn(f"c{i}", F.col("id") + i)
spark.sparkContext.setCheckpointDir("data/shop/checkpoints")
print(plan_lines(long_df), "plan lines before")
checkpointed = long_df.checkpoint()
print(plan_lines(checkpointed), "plan line after")
checkpointed.explain()
files = [f for _, _, names in os.walk("data/shop/checkpoints") for f in names if not f.startswith(".")]
print(len(files), "files in the checkpoint folder")
```

Result:

```
61 plan lines before
1 plan line after
*(1) Scan ExistingRDD[id#605L,c0#606L,c1#607L,c2#608L,c3#609L,c4#610L,c5#611L,c6#612L,c7#613L,c8#614L,c9#615L,c10#616L,c11#617L,c12#618L,c13#619L,c14#620L,c15#621L,c16#622L,c17#623L,c18#624L,c19#625L,c20#626L,c21#627L,c22#628L,c23#629L,... 36 more fields]
4 files in the checkpoint folder
```

</details>

<details>
<summary><b>A7</b> &nbsp;This session allows the driver at most 10 MB of results (spark.driver.maxResultSize). Collect 3 million rows: ...</summary>

**The job fails: the results are bigger than `spark.driver.maxResultSize`.** The driver is a single JVM; pulling a big result into it causes a **driver OOM** (or, with this limit, a clean error). Fixes: aggregate on the executors and bring back only the small result (`count()`, `groupBy(...).agg(...)`); bring back only a few rows (`limit`, `take`, `show`); write big results to files instead of collecting them; avoid `toPandas()` on big data. Raise `spark.driver.memory` (and `maxResultSize`) only when a big result really has to come back. Broadcasting a big table also goes through the driver.

```python
rows = spark.range(3_000_000).selectExpr("id", "repeat('x', 20) AS padding")
try:
    rows.collect()
except Exception as error:
    message = str(error)
    print("collect failed:", message[message.index("Total size"):].splitlines()[0])
print("count on the executors:", rows.count())
```

Result:

```
collect failed: Total size of serialized results of 3 tasks (11.0 MiB) is bigger than spark.driver.maxResultSize (10.0 MiB)
count on the executors: 3000000
```

</details>

<details>
<summary><b>B1</b> &nbsp;What is the median quantity of each status? Show status and median_quantity.</summary>

```python
orders.groupBy("status").agg(F.median("quantity").alias("median_quantity")).orderBy("status").show()
```

</details>

<details>
<summary><b>B2</b> &nbsp;Show the 25th, 50th and 75th percentile of the product prices, as one row with the columns p25, p50 and p75.</summary>

```python
products.agg(F.percentile("price", 0.25).alias("p25"),
             F.percentile("price", 0.5).alias("p50"),
             F.percentile("price", 0.75).alias("p75")).show()
```

</details>

<details>
<summary><b>B3</b> &nbsp;Show the revenue of delivered orders by category and product, with a subtotal for each category and a grand ...</summary>

```python
(orders.filter(F.col("status") == "delivered")
       .join(products, "product_id")
       .rollup("category", "product_name")
       .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue"))
       .orderBy(F.col("category").asc_nulls_last(), F.col("product_name").asc_nulls_last())
       .show())
```

</details>

<details>
<summary><b>B4</b> &nbsp;For each customer, list their order ids as one text, smallest first, like 1001, 1002, 1007, 1015. Leave out ...</summary>

```python
(orders.filter(F.col("customer_id").isNotNull())
       .groupBy("customer_id")
       .agg(F.array_join(F.array_sort(F.collect_list(F.col("order_id").cast("string"))), ", ").alias("order_ids"))
       .orderBy("customer_id")
       .show(truncate=False))
```

</details>

<details>
<summary><b>B5</b> &nbsp;Show the product of each customer's latest order: customer_id and latest_product_id (leave out the guest ...</summary>

```python
(orders.filter(F.col("customer_id").isNotNull())
       .groupBy("customer_id")
       .agg(F.max_by("product_id", "order_date").alias("latest_product_id"))
       .orderBy("customer_id")
       .show())
```

</details>

<details>
<summary><b>B6</b> &nbsp;What share of the delivered revenue does each category bring, in percent (1 decimal)? Show category, revenue ...</summary>

```python
revenue = (orders.filter(F.col("status") == "delivered")
                 .join(products, "product_id")
                 .groupBy("category")
                 .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue")))
(revenue.withColumn("share_pct", F.round(F.col("revenue") / F.sum("revenue").over(Window.partitionBy()) * 100, 1))
        .orderBy(F.col("revenue").desc())
        .show())
```

</details>

<details>
<summary><b>B7</b> &nbsp;Show the second most expensive product of each category (categories with only one product drop out). Columns: ...</summary>

```python
w = Window.partitionBy("category").orderBy(F.col("price").desc())
(products.withColumn("price_rank", F.dense_rank().over(w))
         .filter(F.col("price_rank") == 2)
         .select("category", "product_name", "price")
         .orderBy("category")
         .show())
```

</details>

<details>
<summary><b>B8</b> &nbsp;Which products cost more than the average price of their category? Show category, product_name, price and ...</summary>

```python
w = Window.partitionBy("category")
(products.withColumn("category_avg", F.round(F.avg("price").over(w), 2))
         .filter(F.col("price") > F.col("category_avg"))
         .select("category", "product_name", "price", "category_avg")
         .orderBy("category")
         .show())
```

</details>

<details>
<summary><b>B9</b> &nbsp;For each city, count the customers and how many of them have an email. Customers without a city count under ...</summary>

```python
(customers.groupBy(F.coalesce(F.col("city"), F.lit("Unknown")).alias("city"))
          .agg(F.count("*").alias("customers"), F.count("email").alias("with_email"))
          .orderBy("city")
          .show())
```

</details>

<details>
<summary><b>B10</b> &nbsp;Compare customers with customers_new: which customers were added, removed or changed (a different name, email ...</summary>

```python
old, new = customers.alias("old"), customers_new.alias("new")
same = (F.col("old.name").eqNullSafe(F.col("new.name"))
        & F.col("old.email").eqNullSafe(F.col("new.email"))
        & F.col("old.city").eqNullSafe(F.col("new.city")))
(old.join(new, F.col("old.customer_id") == F.col("new.customer_id"), "full")
    .select(F.coalesce(F.col("old.customer_id"), F.col("new.customer_id")).alias("customer_id"),
            F.when(F.col("old.customer_id").isNull(), "added")
             .when(F.col("new.customer_id").isNull(), "removed")
             .when(~same, "changed").alias("change"))
    .filter(F.col("change").isNotNull())
    .orderBy("customer_id")
    .show())
```

</details>

<details>
<summary><b>B11</b> &nbsp;Make a table of delivered revenue with one row per category and one column per month (Jan, Feb, Mar), with 0 ...</summary>

```python
(orders.filter(F.col("status") == "delivered")
       .join(products, "product_id")
       .withColumn("month", F.date_format("order_date", "MMM"))
       .groupBy("category")
       .pivot("month", ["Jan", "Feb", "Mar"])
       .agg(F.sum(F.col("quantity") * F.col("price")))
       .na.fill(0)
       .orderBy("category")
       .show())
```

</details>